# Support Vector Machine (SVM) Training Flow

Flow:
1. Load raw train / validation / test splits.
2. Apply `DataTransformation` correctly:
   - `transform()` on validation and test data.
3. Optionally load imbalance-resolved training datasets if they already exist.
4. Train multiple SVM experiments.
5. Evaluate on validation data.
6. Tune hyperparameters.
7. Evaluate the selected model on test data.
8. Save the final model and transformer/pipeline.

In [8]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

while not (project_root / "src").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Could not find project root containing 'src'")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\CUFE\Data Science\Diabetes-Prediction


In [9]:
from sklearn.svm import SVC
import pandas as pd
from src.diabetes_prediction.transformation.transformation import DataTransformation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    average_precision_score,  # PR-AUC
    matthews_corrcoef
)


### Load Data


In [10]:
train_transformed = pd.read_csv('../../data/preprocessed/train_preprocessed.csv')
x_train_transformed = train_transformed.drop("diabetes", axis=1)
y_train_transformed = train_transformed["diabetes"]

val = pd.read_csv('../../data/preprocessed/validation_preprocessed.csv')
x_val = val.drop("diabetes", axis=1)
y_val = val["diabetes"]

print(f'x_train_transformed: {train_transformed.shape}')
train_transformed.head()

x_train_transformed: (67139, 21)


,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current,age,bmi,HbA1c_level,blood_glucose_level,glucose_hba1c_interaction,...,age_bmi_interaction,bmi_hba1c_interaction,age_glucose_interaction,gender,hypertension,heart_disease,high_hba1c_flag,senior_flag,cardio_risk_flag,diabetes
0,0.0,0.0,0.0,0.0,0.0,0.612112,-0.637209,0.000000,-0.677966,-0.459103,...,-0.056612,-0.232854,-0.081827,0,0,0,0,0,0,0
1,0.0,0.0,1.0,0.0,0.0,0.737237,0.818605,0.500000,0.000000,0.411609,...,0.658534,1.018393,0.557564,0,0,0,0,0,0,0
2,0.0,0.0,1.0,0.0,0.0,0.399399,-0.229457,0.000000,0.254237,0.382586,...,-0.339001,0.014118,-0.070409,1,0,0,0,0,0,0
3,0.0,1.0,0.0,0.0,0.0,0.874875,1.485271,0.500000,0.254237,0.668865,...,1.258590,1.470922,1.050428,0,0,0,0,1,0,0
4,0.0,0.0,0.0,1.0,0.0,0.687187,-0.395349,-1.642857,1.016949,-0.142480,...,0.148131,-1.008759,1.078972,1,0,0,0,0,0,0


In [11]:
train_ADASYN = pd.read_csv('../../data/imbalance_resolve/ADASYN.csv')
x_train_ADASYN = train_ADASYN.drop("diabetes_target", axis=1)
y_train_ADASYN = train_ADASYN["diabetes_target"]
print(f'train_ADASYN: {train_ADASYN.shape}')

train_smote_enn = pd.read_csv('../../data/imbalance_resolve/train_smote_enn.csv')
x_train_smote_enn = train_smote_enn.drop("diabetes_target", axis=1)
y_train_smote_enn = train_smote_enn["diabetes_target"]
print(f'train_smote_enn: {train_smote_enn.shape}')

train_smote_tomek = pd.read_csv('../../data/imbalance_resolve/train_smote_tomek.csv')
x_train_smote_tomek= train_smote_tomek.drop("diabetes_target", axis=1)
y_train_smote_tomek = train_smote_tomek["diabetes_target"]
print(f'train_smote_tomek: {train_smote_tomek.shape}')

train_smote = pd.read_csv('../../data/imbalance_resolve/train_smote.csv')
x_train_smote = train_smote.drop("diabetes_target", axis=1)
y_train_smote = train_smote["diabetes_target"]
print(f'train_smote: {train_smote.shape}')

train_ADASYN: (122666, 16)
train_smote_enn: (112303, 16)
train_smote_tomek: (121658, 16)
train_smote: (112303, 16)


### Evaluation Metrics
- Accuracy
- **Precision**: When I say diabetic, how often am I right?, TP / (TP + FP)
- **Recall**: How many diabetics did I catch?, TP / (TP + FN) :: **most important**
- **F1-score**
- ROC-AUC
- **PR-AUC**
- **MCC**

`it is more important to us to say that a patient has diabetes when he has, and missing that would be bad that is why recall is most important here`

#### Helper functions for model evaluation

In [16]:
def evaluate_model(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]

    y_pred = (y_proba >= threshold).astype(int)

    results = {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred, zero_division=0),
        "Recall": recall_score(y, y_pred),
        "F1-score": f1_score(y, y_pred),
        "ROC-AUC": roc_auc_score(y, y_proba),
        "PR-AUC": average_precision_score(y, y_proba),
        "MCC": matthews_corrcoef(y, y_pred)
    }    

    return results

def print_results(title, results):
    print(f"========== {title} ==========")
    for k, v in results.items():
        print(f"{k}: {v:.4f}")
    print()


def evaluate_and_store(results_table, name, model, X, y, threshold=0.5):
    results = evaluate_model(model, X, y, threshold=threshold)
    print_results(name, results)

    row = {"Experiment": name}
    row.update(results)
    results_table.append(row)

    

## Experiment 1 — SVM on Transformed Train Data

In [17]:
results_table = []
model_svm_transformed = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight=None,
    probability=True,
    random_state=42,
)

model_svm_transformed.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    results_table,
    "SVM - transformed",
    model_svm_transformed,
    x_val,
    y_val,
)

========== SVM - transformed ==========
Accuracy: 0.9707
Precision: 0.9610
Recall: 0.6973
F1-score: 0.8082
ROC-AUC: 0.9443
PR-AUC: 0.8419
MCC: 0.8048



## Experiment 2 — SVM on Transformed Train Data with Class Weights

In [18]:
model_svm_transformed_w = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42,
)

model_svm_transformed_w.fit(x_train_transformed, y_train_transformed)

evaluate_and_store(
    results_table,
    "SVM - transformed - class_weight balanced",
    model_svm_transformed_w,
    x_val,
    y_val,
)

========== SVM - transformed - class_weight balanced ==========
Accuracy: 0.9582
Precision: 0.8535
Recall: 0.6368
F1-score: 0.7294
ROC-AUC: 0.9693
PR-AUC: 0.8391
MCC: 0.7162



## Experiment 3 — SVM on ADASYN Train Data 


In [19]:
model_svm_adasyn = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight=None,
    probability=True,
    random_state=42,
)
model_svm_adasyn.fit(x_train_ADASYN, y_train_ADASYN)
evaluate_and_store(
    results_table,
    "SVM - ADASYN",
    model_svm_transformed_w,
    x_val,
    y_val,
)


========== SVM - ADASYN ==========
Accuracy: 0.9582
Precision: 0.8535
Recall: 0.6368
F1-score: 0.7294
ROC-AUC: 0.9693
PR-AUC: 0.8391
MCC: 0.7162



## Experiment 4 — SVM on SMOTE Train Data 

In [21]:
model_svm_smote = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight=None,
    probability=True,
    random_state=42,
)
model_svm_smote.fit(x_train_smote, y_train_smote)
evaluate_and_store(
    results_table,
    "SVM - SMOTE",
    model_svm_transformed_w,
    x_val,
    y_val,
)

========== SVM - SMOTE ==========
Accuracy: 0.9582
Precision: 0.8535
Recall: 0.6368
F1-score: 0.7294
ROC-AUC: 0.9693
PR-AUC: 0.8391
MCC: 0.7162



## Experiment 5 — SVM on SMOTE-ENN Train Data 

In [22]:
model_svm_smote_enn = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight=None,
    probability=True,
    random_state=42,
)
model_svm_smote_enn.fit(x_train_smote_enn, y_train_smote_enn)
evaluate_and_store(
    results_table,
    "SVM - SMOTE-ENN",
    model_svm_transformed_w,
    x_val,
    y_val,
)

========== SVM - SMOTE-ENN ==========
Accuracy: 0.9582
Precision: 0.8535
Recall: 0.6368
F1-score: 0.7294
ROC-AUC: 0.9693
PR-AUC: 0.8391
MCC: 0.7162



## Experiment 6 — SVM on SMOTE-TOMEK Train Data 

In [25]:
model_svm_smote_tomek = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    class_weight=None,
    probability=True,
    random_state=42,
)
model_svm_smote_tomek.fit(x_train_smote_tomek, y_train_smote_tomek)
evaluate_and_store(
    results_table,
    "SVM - SMOTE-ENN_tomek",
    model_svm_transformed_w,
    x_val,
    y_val,
)

========== SVM - SMOTE-ENN_tomek ==========
Accuracy: 0.9582
Precision: 0.8535
Recall: 0.6368
F1-score: 0.7294
ROC-AUC: 0.9693
PR-AUC: 0.8391
MCC: 0.7162



## SVM Experiments Evaluation Results

| # | Experiment | Accuracy | Precision | Recall | F1-score | ROC-AUC | PR-AUC | MCC |
|---:|---|---:|---:|---:|---:|---:|---:|---:|
| 1 | SVM - transformed | **0.9707** | **0.9610** | **0.6973** | **0.8082** | 0.9443 | **0.8419** | **0.8048** |
| 2 | SVM - transformed - class_weight balanced | 0.9582 | 0.8535 | 0.6368 | 0.7294 | **0.9693** | 0.8391 | 0.7162 |
| 3 | SVM - ADASYN | 0.9582 | 0.8535 | 0.6368 | 0.7294 | **0.9693** | 0.8391 | 0.7162 |
| 4 | SVM - SMOTE | 0.9582 | 0.8535 | 0.6368 | 0.7294 | **0.9693** | 0.8391 | 0.7162 |
| 5 | SVM - SMOTE-ENN | 0.9582 | 0.8535 | 0.6368 | 0.7294 | **0.9693** | 0.8391 | 0.7162 |
| 6 | SVM - SMOTE-TOMEK | 0.9582 | 0.8535 | 0.6368 | 0.7294 | **0.9693** | 0.8391 | 0.7162 |

### Summary

The best overall model is **SVM - transformed**.

It achieved the highest:

- **Accuracy:** 0.9707
- **Precision:** 0.9610
- **Recall:** 0.6973
- **F1-score:** 0.8082
- **PR-AUC:** 0.8419
- **MCC:** 0.8048

Although the balanced and resampling-based models achieved a higher **ROC-AUC** of **0.9693**, their classification performance was weaker, especially in terms of **Recall**, **F1-score**, and **MCC**.

Therefore, based on the current results, **SVM - transformed** is the preferred model.

## Tunning hyperparameters using x_train_transformed

| Hyperparameter | Meaning | Importance |
|---|---|---|
| `kernel` | Defines the shape of the decision boundary. | Controls whether the model learns a linear or non-linear relationship. |
| `C` | Controls the penalty for misclassification. | Balances underfitting and overfitting. |
| `gamma` | Controls how far the influence of each training point reaches. | Important for RBF kernel; affects boundary complexity. |
| `class_weight` | Assigns different weights to classes. | Helps handle imbalanced data by giving more importance to the minority class. |

In [26]:
tuning_results = []

param_grid = [
    {"kernel": "rbf", "C": 0.1, "gamma": "scale", "class_weight": "balanced"},
    {"kernel": "rbf", "C": 1.0, "gamma": "scale", "class_weight": "balanced"},
    {"kernel": "rbf", "C": 10.0, "gamma": "scale", "class_weight": "balanced"},
    {"kernel": "rbf", "C": 1.0, "gamma": 0.01, "class_weight": "balanced"},
    {"kernel": "rbf", "C": 1.0, "gamma": 0.1, "class_weight": "balanced"},
    {"kernel": "linear", "C": 0.1, "gamma": "scale", "class_weight": "balanced"},
    {"kernel": "linear", "C": 1.0, "gamma": "scale", "class_weight": "balanced"},
]

trained_tuned_models = {}

for params in param_grid:
    name = (
        f"SVM tuned | kernel={params['kernel']} | "
        f"C={params['C']} | gamma={params['gamma']} | "
        f"class_weight={params['class_weight']}"
    )

    model = SVC(
        kernel=params["kernel"],
        C=params["C"],
        gamma=params["gamma"],
        class_weight=params["class_weight"],
        probability=True,
        random_state=42,
    )

    model.fit(x_train_transformed,y_train_transformed)
    trained_tuned_models[name] = model

    results = evaluate_model(model, x_val, y_val)

    row = {"Experiment": name}
    row.update(results)
    tuning_results.append(row)

tuning_results_df = pd.DataFrame(tuning_results)
tuning_results_df = tuning_results_df.sort_values(
    by=["Recall", "F1-score", "ROC-AUC"],
    ascending=False
)

tuning_results_df

,Experiment,Accuracy,Precision,Recall,F1-score,ROC-AUC,PR-AUC,MCC
0,SVM tuned | kernel=rbf | C=0.1 | gamma=scale |...,0.945506,0.693959,0.686321,0.690119,0.962259,0.776715,0.660258
3,SVM tuned | kernel=rbf | C=1.0 | gamma=0.01 | ...,0.956697,0.824176,0.648585,0.725913,0.962902,0.817316,0.708715
6,SVM tuned | kernel=linear | C=1.0 | gamma=scal...,0.958574,0.850622,0.644654,0.733453,0.962531,0.821744,0.719499
5,SVM tuned | kernel=linear | C=0.1 | gamma=scal...,0.958504,0.851929,0.642296,0.732407,0.962519,0.822477,0.718728
1,SVM tuned | kernel=rbf | C=1.0 | gamma=scale |...,0.958226,0.853530,0.636792,0.729401,0.969337,0.839083,0.716203
2,SVM tuned | kernel=rbf | C=10.0 | gamma=scale ...,0.965177,0.958383,0.633648,0.762896,0.969905,0.853046,0.763553
4,SVM tuned | kernel=rbf | C=1.0 | gamma=0.1 | c...,0.958574,0.861111,0.633648,0.730072,0.970145,0.843140,0.717981


In [27]:
best_experiment_name = tuning_results_df.iloc[0]["Experiment"]
best_svm_model = trained_tuned_models[best_experiment_name]

print("Best experiment:")
print(best_experiment_name)

evaluate_model(best_svm_model, x_val, y_val)

Best experiment:
SVM tuned | kernel=rbf | C=0.1 | gamma=scale | class_weight=balanced


{'Accuracy': 0.9455063599082505,
 'Precision': 0.6939586645468998,
 'Recall': 0.6863207547169812,
 'F1-score': 0.6901185770750988,
 'ROC-AUC': 0.9622592655200608,
 'PR-AUC': 0.7767150770530534,
 'MCC': 0.6602582369123956}

## Save the model

In [28]:
import joblib

joblib.dump(best_svm_model, "../../models/SVM_model.pkl")


['../../models/SVM_model.pkl']